# Q_GRU — Quantile GRU for RUL Prediction

GRU — reset + update gates; simpler than LSTM, often more stable.
Fewer parameters than LSTM → lower overfitting risk on small datasets.

**Structure:** Same as DL notebooks (MLP.ipynb, GRU.ipynb, etc.) but:
- Loss: **Pinball loss** instead of NASA asymmetric loss
- Output: **3 neurons** (Q10, Q50, Q90) instead of 1
- Evaluation: Q50 used for RMSE/NASA; Q10-Q90 interval for uncertainty quantification

## 1. Imports & Setup

In [ ]:
import sys, os
from pathlib import Path

ROOT = Path(os.getcwd()).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src.models.deep_learning import (
    NASALoss, PinballLoss, RULDataset,
    load_data, select_features, build_windows, engine_split, make_loaders,
    train_quantile_model, predict_quantiles, evaluate_quantile_model,
    plot_quantile_predictions, plot_loss_curves,
)
from src.evaluation.metrics import save_model_results
from src.evaluation.metrics import (
    pinball_loss_by_quantile, reliability_diagram, interval_coverage_by_rul_bucket
)

WINDOW_SIZE = 30
RUL_CAP     = 125
BATCH_SIZE  = 128
EPOCHS      = 50
LR          = 1e-3
QUANTILES   = [0.1, 0.5, 0.9]
RANDOM_SEED = 42

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DEVICE = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else 'cpu'
)
print(f'Device: {DEVICE}')
print(f'Predicting quantiles: {QUANTILES}')

## 2. Load Data

In [ ]:
train_df, test_df = load_data()
FEAT_COLS  = select_features(train_df)
N_FEATURES = len(FEAT_COLS)

## 3. Build Sliding Windows (30-cycle, engine-split 80/20)

In [ ]:
X_train, y_train, X_val, y_val = engine_split(train_df, FEAT_COLS)
X_test,  y_test  = build_windows(test_df, FEAT_COLS, is_test=True)
print(f'X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}')

## 4. DataLoaders

In [ ]:
train_loader, val_loader, test_loader = make_loaders(
    X_train, y_train, X_val, y_val, X_test, y_test
)

## 5. Model Definition — QuantileGRU

In [ ]:
class QuantileGRU(nn.Module):
    def __init__(self, n_features, hidden_size=64, n_layers=2, n_quantiles=3, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(n_features, hidden_size, n_layers, batch_first=True,
                          dropout=dropout if n_layers > 1 else 0.0)
        self.fc  = nn.Linear(hidden_size, n_quantiles)
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])


model = QuantileGRU(n_features=N_FEATURES)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: {model.__class__.__name__}')
print(f'Trainable parameters: {total_params:,}')

## 6. Training with Pinball Loss

In [ ]:
model, train_losses, val_losses = train_quantile_model(
    model        = model,
    train_loader = train_loader,
    val_loader   = val_loader,
    quantiles    = QUANTILES,
    epochs       = EPOCHS,
    lr           = LR,
    model_name   = 'Q_GRU',
)
plot_loss_curves(train_losses, val_losses, model_name='Q_GRU')

## 7. Evaluation — Point Metrics (Q50) + Interval Metrics

In [ ]:
y_true, q10, q50, q90 = predict_quantiles(model, test_loader)
results, width, coverage = evaluate_quantile_model(y_true, q10, q50, q90, model_name='Q_GRU')
plot_quantile_predictions(y_true, q10, q50, q90, model_name='Q_GRU')

## 8. Calibration Metrics

Addresses critic: *'No calibration metrics — coverage probability, pinball loss missing'*

In [ ]:
import numpy as np

# Pinball loss per quantile
preds_matrix = np.stack([q10, q50, q90], axis=1)
pinball_df = pinball_loss_by_quantile(y_true, preds_matrix, QUANTILES, model_name='Q_GRU')

# Reliability diagram
all_q_preds = {0.1: q10, 0.5: q50, 0.9: q90}
reliability_diagram(y_true, all_q_preds, title='Reliability Diagram — Q_GRU')

# Interval coverage by RUL bucket
bucket_df = interval_coverage_by_rul_bucket(y_true, q10, q90, model_name='Q_GRU')

## 9. Save Results to CSV

In [ ]:
save_model_results(
    model_name = 'Q_GRU',
    model_type = 'quantile',
    y_true     = y_true,
    y_pred     = q50,
    q10        = q10,
    q90        = q90,
)
print('Results saved to results/all_model_results.csv')